## Deployment & Interpretation

### 1. Persiapan

#### 1.1. Import Library

In [ ]:
import os
import json
import pandas as pd
from config import (
  CLEANED_PROVINCES_CSV,
  CLUSTERED_REGENCIES_CSV,
  SELECTED_FEATURES,
  SEED_SQL
)

#### 1.2. Persiapan Data

In [ ]:
df_prov = pd.read_csv(CLEANED_PROVINCES_CSV)
df_reg = pd.read_csv(CLUSTERED_REGENCIES_CSV)

#### 1.3. Parameter Interpretasi AI

In [ ]:
MODEL = "@cf/openai/gpt-oss-120b"
MAX_TOKENS = 3000

### 2. Interpretasi Karakteristik & Tipologi Klaster

#### 2.1. Profil Rata-Rata Fitur Tiap Klaster

In [ ]:
active_features = [c for c in SELECTED_FEATURES if c in df_reg.columns]
profile_df = df_reg.groupby('cluster_label')[active_features].mean().round(2)

profile_text = ""
labels_map = {}

for label, group in df_reg.groupby('cluster_label'):
  avg_koperasi = group['total_koperasi'].mean()
  avg_transaksi = group['nilai_transaksi'].mean()
  avg_nib = group['rasio_nib'].mean()
  
  if avg_transaksi > df_reg['nilai_transaksi'].mean() and avg_koperasi > df_reg['total_koperasi'].mean():
    tipologi = "Klaster Sentra Ekonomi Utama (Skala Usaha & Transaksi Sangat Tinggi)"
  elif avg_nib > 80 and avg_koperasi > df_reg['total_koperasi'].mean():
    tipologi = "Klaster Koperasi Berkembang (Kepatuhan Formalitas & Skala Tinggi)"
  elif avg_transaksi > df_reg['nilai_transaksi'].mean():
    tipologi = "Klaster Potensi Transaksi Produktif (Aktivitas Ekonomi Signifikan)"
  else:
    tipologi = "Klaster Koperasi Rintisan (Perlu Akselerasi Kelembagaan & Usaha)"

  labels_map[str(label)] = f"Klaster {label}: {tipologi}"
  
  profile_text += f"\n### Klaster {label} ({len(group)} Kabupaten/Kota)\n"
  profile_text += f"**Tipologi**: {tipologi}\n\n"
  for col in active_features:
    m = group[col].mean()
    lbl = col.replace('_', ' ').title()
    if "nilai" in col or "simpanan" in col:
      profile_text += f"- Rata-rata {lbl}: Rp {m:,.2f}\n"
    elif "rasio" in col:
      profile_text += f"- Rata-rata {lbl}: {m:.2f}%\n"
    else:
      profile_text += f"- Rata-rata {lbl}: {m:,.2f}\n"

print("Tipologi Klaster Terbentuk:")
for k, v in labels_map.items():
  print(f"[{k}] -> {v}")

#### 2.2. Laporan Lengkap Interpretasi Klaster

In [ ]:
full_report_text = f"""# Laporan Interpretasi Hasil Klasterisasi Koperasi Desa/Kelurahan (SIMKOPDES)

Berdasarkan hasil analisis klastering algoritma K-Means terhadap indikator kelembagaan, kepatuhan legalitas (NIB, NPWP, RAT), dan kinerja transaksi keuangan koperasi di 514 Kabupaten/Kota di Indonesia, diperoleh segmentasi tipologi wilayah sebagai berikut:

{profile_text}

## Rekomendasi Kebijakan & Intervensi:
1. **Klaster Skala/Transaksi Tinggi**: Penguatan integrasi rantai pasok industri dan diversifikasi produk bernilai tambah.
2. **Klaster Legalitas Berkembang**: Fasilitasi kemitraan perbankan/fintech dan peningkatan literasi RAT tahunan.
3. **Klaster Rintisan**: Pendampingan intensif kelembagaan, digitalisasi pembukuan, dan program pembiayaan awal.
"""

print(full_report_text[:600] + "...")

### 3. Pembentukan Data Seed SQL Cloudflare D1

#### 3.1. Format Helper SQL Sanitizer

In [ ]:
def sql_val(val):
  if pd.isna(val) or val is None:
    return "NULL"
  if isinstance(val, (int, float)):
    return str(val)
  escaped = str(val).replace("'", "''")
  return f"'{escaped}'"

#### 3.2. Generate Pernyataan SQL Insert

In [ ]:
sql_lines = [
  "-- Cloudflare D1 SQL Seed Generated Automatically by Pipeline\n",
  "-- Insert Provinces"
]

for idx, r in df_prov.iterrows():
  p_id = sql_val(r.get('province_id', r.get('no', idx + 1)))
  p_name = sql_val(r['province_name'])
  lat = sql_val(r.get('latitude', 0.0))
  lon = sql_val(r.get('longitude', 0.0))
  sql_lines.append(
    f"INSERT OR REPLACE INTO provinces (id, province_name, total_koperasi, koperasi_nib, koperasi_npwp, koperasi_rat, rasio_nib, rasio_npwp, rasio_rat, simpanan_pokok, simpanan_wajib, volume_transaksi, nilai_transaksi, latitude, longitude) VALUES ({p_id}, {p_name}, {r['total_koperasi']}, {r['koperasi_nib']}, {r['koperasi_npwp']}, {r['koperasi_rat']}, {r['rasio_nib']}, {r['rasio_npwp']}, {r['rasio_rat']}, {r['simpanan_pokok']}, {r['simpanan_wajib']}, {r['volume_transaksi']}, {r['nilai_transaksi']}, {lat}, {lon});"
  )

sql_lines.append("\n-- Insert Regencies")
for idx, r in df_reg.iterrows():
  r_id = sql_val(idx + 1)
  p_id = sql_val(r.get('province_id', 1))
  r_name = sql_val(r['regency_name'])
  lat = sql_val(r.get('latitude', 0.0))
  lon = sql_val(r.get('longitude', 0.0))
  sql_lines.append(
    f"INSERT OR REPLACE INTO regencies (id, province_id, regency_name, total_koperasi, koperasi_nib, koperasi_npwp, koperasi_rat, rasio_nib, rasio_npwp, rasio_rat, simpanan_pokok, simpanan_wajib, volume_transaksi, nilai_transaksi, cluster_label, latitude, longitude) VALUES ({r_id}, {p_id}, {r_name}, {r['total_koperasi']}, {r['koperasi_nib']}, {r['koperasi_npwp']}, {r['koperasi_rat']}, {r['rasio_nib']}, {r['rasio_npwp']}, {r['rasio_rat']}, {r['simpanan_pokok']}, {r['simpanan_wajib']}, {r['volume_transaksi']}, {r['nilai_transaksi']}, {r['cluster_label']}, {lat}, {lon});"
  )

sql_lines.append("\n-- Insert AI Interpretation Report")
report_escaped = sql_val(full_report_text)
labels_escaped = sql_val(json.dumps(labels_map))
sql_lines.append(
  f"INSERT OR REPLACE INTO ai_report (id, report_text, labels_json) VALUES (1, {report_escaped}, {labels_escaped});"
)

os.makedirs(os.path.dirname(SEED_SQL), exist_ok=True)
with open(SEED_SQL, 'w', encoding='utf-8') as f:
  f.write("\n".join(sql_lines))

### 4. Verifikasi Output SQL Seed

In [ ]:
file_size_kb = round(os.path.getsize(SEED_SQL) / 1024, 2)
print(f"File SQL Seed berhasil dibuat : {SEED_SQL}")
print(f"Ukuran File                   : {file_size_kb} KB")
print(f"Total Entri Provinsi          : {len(df_prov)}")
print(f"Total Entri Kabupaten/Kota    : {len(df_reg)}")

#### 4.1. Contoh 10 Baris Pertama File SQL

In [ ]:
with open(SEED_SQL, 'r', encoding='utf-8') as f:
  sample_lines = [f.readline().strip() for _ in range(10)]

for line in sample_lines:
  print(line)